# 01 — Data integrity and chronological splits

**Goal:** establish the exact auction population before any model is trained. This stage
preserves the paper's millisecond timestamp, consolidates joined event rows to one Bid ID,
removes temporal overlap, and writes immutable stage inputs with hashes.


## Paper anchor and extension

Zhang et al. define Bid ID as the join key across event logs and specify a 17-digit
timestamp. The paper also notes duplicate click events. We therefore consolidate outcome
records rather than treating each row as an independent auction. The strict non-overlap
rule is an extension: it removes advertiser 2997's same-day train/test overlap.


In [ ]:
from pathlib import Path
import os

# Run correctly whether Jupyter starts in the project root or in notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
ARTIFACT_ROOT = Path(os.getenv("IPINYOU_ARTIFACT_ROOT", PROJECT_ROOT / "artifacts"))
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT}")


In [ ]:
from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(*objects):
        for obj in objects:
            print(obj)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

RANDOM_STATE = 42
NOTEBOOK_SCHEMA_VERSION = "2.2"
TARGET_ADVERTISERS = [1458, 2997]

USE_KAGGLEHUB = False
KAGGLE_DATASET_SLUG = "pleaseholdme/ipinyou"
DATA_ROOT_OVERRIDE = os.getenv("IPINYOU_DATA_ROOT")

MIN_TRAIN_ROWS, MIN_TRAIN_CLICKS = 50_000, 100
MIN_TEST_ROWS, MIN_TEST_CLICKS = 20_000, 20
FIT_FRAC, CAL_FRAC = 0.70, 0.15
TUNING_WINDOW_MAX_ROWS, TUNING_EVAL_FRAC = 600_000, 0.20
STRICT_TEMPORAL_HOLDOUT = True
ROUND_OVERLAP_CUTOFF_TO_NEXT_HOUR = True

SMOKE_TEST = os.getenv("IPINYOU_SMOKE_TEST", "0") == "1"
if SMOKE_TEST:
    MIN_TRAIN_ROWS, MIN_TRAIN_CLICKS = 1_000, 10
    MIN_TEST_ROWS, MIN_TEST_CLICKS = 500, 5
    TUNING_WINDOW_MAX_ROWS = 4_000

display(pd.Series({
    "schema_version": NOTEBOOK_SCHEMA_VERSION,
    "smoke_test": SMOKE_TEST,
    "target_advertisers": TARGET_ADVERTISERS,
    "strict_temporal_holdout": STRICT_TEMPORAL_HOLDOUT,
    "preserve_timestamp_milliseconds": True,
}, name="value").to_frame())


## 3. Data loading, deduplication, and integrity audit

The official iPinYou schema describes `bidid` as a unique impression-opportunity identifier.
Some distributed mirrors nevertheless contain multiple log records for the same Bid ID. A
whole-row equality test is too strict because joined outcome and tag fields can legitimately
differ across those records. This version therefore constructs **one auction per Bid ID** with
a deterministic, outcome-preserving rule:

- take the earliest request row for timestamp and pre-decision features;
- use `max(click)` so a later click record is not discarded;
- use `max(payprice)` as a conservative clearing-cost reconciliation;
- store the sorted union of user tags;
- fail on advertiser disagreement within a Bid ID or cross-split Bid ID overlap.

Every field-level disagreement is counted and displayed. This makes the consolidation visible
without treating benign joined-record differences as independent auctions or fatal corruption.

Null/non-finite clearing prices, invalid clicks, advertiser mismatches, and timestamp failures are
checked on the source rows before consolidation and are hard failures. Operational hour and
weekday are always re-derived from the selected timestamp.
The full SHA-256 hashes are displayed so an executed result can be tied to exact input bytes.


In [ ]:
COLUMN_ALIASES = {
    "bidid": "bidid", "bid_id": "bidid", "timestamp": "timestamp",
    "logtype": "logtype", "log_type": "logtype", "ipinyouid": "ipinyouid",
    "userid": "ipinyouid", "useragent": "useragent", "user_agent": "useragent",
    "ip": "ip", "IP": "ip", "region": "region", "city": "city",
    "adexchange": "adexchange", "ad_exchange": "adexchange", "domain": "domain",
    "url": "url", "urlid": "urlid", "slotid": "slotid", "slotwidth": "slotwidth",
    "slotheight": "slotheight", "slotvisibility": "slotvisibility",
    "slotformat": "slotformat", "slotprice": "slotprice", "creative": "creative",
    "bidprice": "bidprice", "payprice": "payprice", "keypage": "keypage",
    "advertiser": "advertiser", "usertag": "usertag", "click": "click",
    "weekday": "weekday", "hour": "hour",
}

REQUIRED_COLUMNS = {
    "click", "timestamp", "weekday", "hour", "region", "city", "adexchange",
    "useragent", "slotwidth", "slotheight", "slotvisibility", "slotformat",
    "slotprice", "creative", "payprice", "advertiser", "bidid",
}


def contains_campaign_files(folder: Path) -> bool:
    return (
        folder.is_dir()
        and (folder / "train.log.txt").is_file()
        and (folder / "test.log.txt").is_file()
    )


def discover_campaign_dirs(root: Path) -> dict[int, Path]:
    """Find advertiser directories; fail later if declared advertisers are ambiguous/missing."""
    root = Path(root).expanduser().resolve()
    found = {}
    if contains_campaign_files(root) and root.name.isdigit():
        found[int(root.name)] = root
    if root.exists():
        for path in sorted(root.iterdir()):
            if contains_campaign_files(path) and path.name.isdigit():
                found[int(path.name)] = path
    if not found and root.exists():
        for train_path in sorted(root.rglob("train.log.txt")):
            folder = train_path.parent
            if folder.name.isdigit() and (folder / "test.log.txt").is_file():
                advertiser = int(folder.name)
                if advertiser in found and found[advertiser] != folder:
                    raise RuntimeError(f"Multiple folders found for advertiser {advertiser}.")
                found[advertiser] = folder
    return dict(sorted(found.items()))


def make_smoke_dataset(root: Path) -> Path:
    """Create two deterministic campaigns, including one overlapping raw test period."""
    root.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(123)
    specs = [
        (1001, 0.00, 0.0, "2013-06-01", "2013-06-12"),
        (1002, 0.35, 8.0, "2013-06-01", "2013-06-08"),
    ]
    for advertiser, ctr_shift, cost_shift, train_start, test_start in specs:
        advertiser_dir = root / str(advertiser)
        advertiser_dir.mkdir(exist_ok=True)
        for split, n_rows, start, n_days in [
            ("train", 6_000, train_start, 8),
            ("test", 2_500, test_start, 4),
        ]:
            t0 = pd.Timestamp(start)
            seconds = rng.integers(0, n_days * 24 * 3600, n_rows)
            event_time = t0 + pd.to_timedelta(seconds, unit="s")
            hour = event_time.hour.to_numpy()
            exchange = rng.choice(["1", "2", "3"], n_rows, p=[0.45, 0.35, 0.20])
            slotprice = rng.gamma(2.0, 18.0, n_rows)
            width = rng.choice([300, 728, 160], n_rows)
            height = np.where(width == 728, 90, np.where(width == 160, 600, 250))
            logit = (
                -5.0 + ctr_shift + 0.45 * (hour >= 18)
                + 0.25 * (exchange == "2") + 0.15 * np.log1p(slotprice) / 4
            )
            click = rng.binomial(1, 1 / (1 + np.exp(-logit)))
            payprice = np.maximum(
                1,
                35 + cost_shift + 8 * (exchange == "3")
                + 0.18 * slotprice + rng.normal(0, 10, n_rows),
            ).round().astype(int)
            frame = pd.DataFrame(
                {
                    "click": click, "weekday": event_time.weekday, "hour": hour,
                    "bidid": [f"{advertiser}-{split}-{i}" for i in range(n_rows)],
                    "timestamp": [x.strftime("%Y%m%d%H%M%S") + "000" for x in event_time],
                    "logtype": 1, "ipinyouid": "u",
                    "useragent": rng.choice(
                        ["windows_chrome", "android_chrome", "ios_safari"], n_rows
                    ),
                    "IP": "0.0.0.0", "region": rng.choice([1, 2, 3, 4], n_rows),
                    "city": rng.choice([10, 20, 30], n_rows), "adexchange": exchange,
                    "domain": "d", "url": "u", "urlid": "", "slotid": "s",
                    "slotwidth": width, "slotheight": height,
                    "slotvisibility": rng.choice(["FirstView", "SecondView"], n_rows),
                    "slotformat": rng.choice(["Fixed", "Pop"], n_rows),
                    "slotprice": slotprice.round(2),
                    "creative": rng.choice(["c1", "c2", "c3"], n_rows),
                    "bidprice": 300, "payprice": payprice, "keypage": "k",
                    "advertiser": advertiser,
                    "usertag": rng.choice(["10006,10063", "10024", ""], n_rows),
                }
            )
            # Exercise the duplicate resolver in every smoke run. These repeat records
            # vary an outcome, tag ordering, cost, and timestamp while retaining one Bid ID.
            repeats = frame.iloc[:4].copy().reset_index(drop=True)
            repeats.loc[0, "click"] = 1
            repeats.loc[1, "usertag"] = "10063,10006"
            repeats.loc[2, "payprice"] = int(repeats.loc[2, "payprice"]) + 5
            later = pd.to_datetime(
                str(repeats.loc[3, "timestamp"])[:14], format="%Y%m%d%H%M%S"
            ) + pd.Timedelta(seconds=5)
            repeats.loc[3, "timestamp"] = later.strftime("%Y%m%d%H%M%S") + "000"
            frame = pd.concat([frame, repeats], ignore_index=True)
            frame.to_csv(advertiser_dir / f"{split}.log.txt", sep="\t", index=False)
    return root


def resolve_data_root() -> Path:
    if SMOKE_TEST:
        smoke_root = Path(os.getenv("IPINYOU_SMOKE_ROOT", Path.cwd() / ".tmp_ipinyou_smoke"))
        return make_smoke_dataset(smoke_root)
    if DATA_ROOT_OVERRIDE:
        path = Path(DATA_ROOT_OVERRIDE).expanduser()
        if path.exists():
            return path.resolve()
    if USE_KAGGLEHUB:
        import kagglehub
        return Path(kagglehub.dataset_download(KAGGLE_DATASET_SLUG)).resolve()
    candidates = [
        Path.cwd() / "data" / "ipinyou", Path.cwd() / "ipinyou",
        Path.cwd() / "data", Path.cwd(), Path("/mnt/data/ipinyou"), Path("/mnt/data"),
    ]
    for candidate in candidates:
        if candidate.exists() and discover_campaign_dirs(candidate):
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find advertiser folders. Set IPINYOU_DATA_ROOT or enable USE_KAGGLEHUB."
    )


def read_ipinyou(path: Path) -> pd.DataFrame:
    # Preserve the 17-digit millisecond timestamp and Bid ID exactly; numeric inference can
    # silently lose timestamp precision in mirrors containing null-like values.
    protected_types = {"timestamp": "string", "bidid": "string", "bid_id": "string"}
    frame = pd.read_csv(path, sep="\t", low_memory=False, dtype=protected_types)
    if frame.shape[1] == 1:
        frame = pd.read_csv(path, sep=",", low_memory=False, dtype=protected_types)
    frame.columns = [
        COLUMN_ALIASES.get(str(column).strip(), str(column).strip().lower())
        for column in frame.columns
    ]
    return frame


def parse_timestamp(series: pd.Series) -> pd.Series:
    # The paper specifies yyyyMMddHHmmssSSS. Keep milliseconds; discarding them creates
    # artificial timestamp ties and can collapse chronological tuning boundaries.
    values = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    values = values.str.extract(r"^(\d{14,17})$", expand=False)
    values = values.str.pad(width=17, side="right", fillchar="0")
    return pd.to_datetime(values, format="%Y%m%d%H%M%S%f", errors="coerce")


def basic_clean(frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize types and derive every operational time field from timestamp."""
    data = frame.copy()
    data["event_time"] = parse_timestamp(data["timestamp"])
    for column in [
        "click", "payprice", "slotprice", "slotwidth", "slotheight",
        "hour", "weekday", "advertiser",
    ]:
        if column in data:
            data[column] = pd.to_numeric(data[column], errors="coerce")
    data["source_hour"] = data["hour"]
    data["source_weekday"] = data["weekday"]
    data["hour"] = data["event_time"].dt.hour
    data["weekday"] = data["event_time"].dt.weekday
    return data


def _sorted_tag_union(values: pd.Series) -> str:
    """Canonical union for comma-separated user-tag records belonging to one auction."""
    tags = set()
    for value in values.dropna().astype(str):
        tags.update(tag.strip() for tag in value.split(",") if tag.strip())
    return ",".join(sorted(tags))


def consolidate_bidids(frame: pd.DataFrame):
    """Resolve repeated log records to one auction and return field-level diagnostics."""
    data = frame.copy()
    bid_text = data["bidid"].astype("string").str.strip()
    null_mask = bid_text.isna() | bid_text.fillna("").eq("")
    data["__bid_key"] = bid_text.fillna("__NULL_BIDID__")
    data["__source_order"] = np.arange(len(data), dtype=np.int64)

    duplicate_mask = data["__bid_key"].duplicated(keep=False)
    duplicate_rows = data.loc[duplicate_mask].copy()
    duplicate_excess_rows = int(data["__bid_key"].duplicated(keep="first").sum())
    duplicate_groups = int(duplicate_rows["__bid_key"].nunique()) if len(duplicate_rows) else 0

    profile_rows = []
    disagreement_id_sets = []
    hard_identity_ids = set()
    if len(duplicate_rows):
        grouped = duplicate_rows.groupby("__bid_key", sort=False, dropna=False)
        ignored = {"bidid", "__bid_key", "__source_order", "source_hour", "source_weekday"}
        for column in sorted(set(duplicate_rows.columns) - ignored):
            counts = grouped[column].nunique(dropna=False)
            disagreeing = set(counts.index[counts > 1].astype(str))
            if disagreeing:
                disagreement_id_sets.append(disagreeing)
                if column == "advertiser":
                    hard_identity_ids.update(disagreeing)
                resolution = {
                    "click": "maximum (outcome preserving)",
                    "payprice": "maximum (conservative cost)",
                    "usertag": "sorted tag union",
                    "event_time": "earliest request row",
                    "timestamp": "earliest request row",
                }.get(column, "earliest request row")
                profile_rows.append(
                    {
                        "field": column,
                        "duplicate_bidids_with_disagreement": len(disagreeing),
                        "resolution": resolution,
                        "hard_failure": column == "advertiser",
                    }
                )

    # Stable sorting makes the selected pre-decision row deterministic even for tied times.
    ordered = data.sort_values(
        ["__bid_key", "event_time", "__source_order"],
        kind="mergesort",
        na_position="last",
    )
    cleaned = ordered.drop_duplicates("__bid_key", keep="first").copy()

    if len(duplicate_rows):
        grouped = duplicate_rows.groupby("__bid_key", sort=False, dropna=False)
        click_max = grouped["click"].max()
        payprice_max = grouped["payprice"].max()
        duplicate_keys = set(duplicate_rows["__bid_key"].astype(str))
        is_reconciled = cleaned["__bid_key"].astype(str).isin(duplicate_keys)
        cleaned.loc[is_reconciled, "click"] = cleaned.loc[
            is_reconciled, "__bid_key"
        ].map(click_max)
        cleaned.loc[is_reconciled, "payprice"] = cleaned.loc[
            is_reconciled, "__bid_key"
        ].map(payprice_max)
        if "usertag" in cleaned:
            tag_union = grouped["usertag"].apply(_sorted_tag_union)
            cleaned.loc[is_reconciled, "usertag"] = cleaned.loc[
                is_reconciled, "__bid_key"
            ].map(tag_union)

    any_disagreement_ids = set().union(*disagreement_id_sets) if disagreement_id_sets else set()
    profile = pd.DataFrame(
        profile_rows,
        columns=[
            "field", "duplicate_bidids_with_disagreement", "resolution", "hard_failure"
        ],
    )
    cleaned = cleaned.drop(columns=["__bid_key", "__source_order"]).reset_index(drop=True)
    return cleaned, {
        "null_bidids": int(null_mask.sum()),
        "duplicate_bidid_groups": duplicate_groups,
        "duplicate_bidid_excess_rows": duplicate_excess_rows,
        "duplicate_bidids_with_any_disagreement": len(any_disagreement_ids),
        "hard_identity_conflicts": len(hard_identity_ids),
        "click_disagreements": int(
            profile.loc[profile["field"].eq("click"), "duplicate_bidids_with_disagreement"].sum()
        ) if len(profile) else 0,
        "payprice_disagreements": int(
            profile.loc[profile["field"].eq("payprice"), "duplicate_bidids_with_disagreement"].sum()
        ) if len(profile) else 0,
        "timestamp_disagreements": int(
            profile.loc[profile["field"].eq("event_time"), "duplicate_bidids_with_disagreement"].sum()
        ) if len(profile) else 0,
        "deduplicated_rows": len(cleaned),
    }, profile


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(block_size):
            digest.update(chunk)
    return digest.hexdigest()


DATA_ROOT = resolve_data_root()
all_campaign_dirs = discover_campaign_dirs(DATA_ROOT)
if SMOKE_TEST:
    CAMPAIGN_DIRS = {k: v for k, v in all_campaign_dirs.items() if k in [1001, 1002]}
else:
    missing = [advertiser for advertiser in TARGET_ADVERTISERS if advertiser not in all_campaign_dirs]
    assert not missing, f"Missing declared advertiser folders: {missing}"
    CAMPAIGN_DIRS = {advertiser: all_campaign_dirs[advertiser] for advertiser in TARGET_ADVERTISERS}

assert len(CAMPAIGN_DIRS) == 2, "This notebook is intentionally a two-advertiser study."
print(f"Data root: {DATA_ROOT}")
print(f"Campaigns locked for this run: {list(CAMPAIGN_DIRS)}")


In [ ]:
def strict_holdout(train_frame: pd.DataFrame, test_frame: pd.DataFrame):
    """Retain only test rows strictly later than the latest cleaned training timestamp."""
    train_end = train_frame["event_time"].max()
    raw_test_start = test_frame["event_time"].min()
    if pd.isna(train_end) or pd.isna(raw_test_start):
        raise ValueError("Training and test data must contain valid timestamps.")
    overlap_detected = bool(raw_test_start <= train_end)
    if not STRICT_TEMPORAL_HOLDOUT or not overlap_detected:
        cutoff = raw_test_start
    elif ROUND_OVERLAP_CUTOFF_TO_NEXT_HOUR:
        cutoff = train_end.ceil("h")
        if cutoff <= train_end:
            cutoff += pd.Timedelta(hours=1)
    else:
        cutoff = train_end + pd.Timedelta(nanoseconds=1)
    if STRICT_TEMPORAL_HOLDOUT and overlap_detected:
        keep = test_frame["event_time"] >= cutoff
        removed = int((~keep).sum())
        retained = test_frame.loc[keep].sort_values("event_time").copy()
    else:
        removed = 0
        retained = test_frame.sort_values("event_time").copy()
    return retained, cutoff, removed


def campaign_audit(advertiser: int, folder: Path):
    train_path = folder / "train.log.txt"
    test_path = folder / "test.log.txt"
    train_source = read_ipinyou(train_path)
    test_source = read_ipinyou(test_path)
    missing_train = sorted(REQUIRED_COLUMNS - set(train_source.columns))
    missing_test = sorted(REQUIRED_COLUMNS - set(test_source.columns))
    if missing_train or missing_test:
        return {
            "advertiser": advertiser,
            "eligible": False,
            "reason": f"missing columns train={missing_train}, test={missing_test}",
        }, None, None, pd.DataFrame()

    train_raw = basic_clean(train_source)
    test_raw = basic_clean(test_source)
    train, train_dedup, train_profile = consolidate_bidids(train_raw)
    test, test_dedup, test_profile = consolidate_bidids(test_raw)

    def bad_clicks(data):
        return int((~data["click"].isin([0, 1])).sum())

    def invalid_costs(data):
        values = pd.to_numeric(data["payprice"], errors="coerce")
        return int((values.isna() | ~np.isfinite(values) | (values < 0)).sum())

    def advertiser_mismatches(data):
        values = pd.to_numeric(data["advertiser"], errors="coerce")
        return int((values != advertiser).sum())

    strict_test, strict_cutoff, overlap_rows_removed = strict_holdout(train, test)
    strict_cross_bid_overlap = len(
        set(train["bidid"].astype(str)) & set(strict_test["bidid"].astype(str))
    ) if len(strict_test) else 0

    train_end = train["event_time"].max()
    chronology_ok = bool(
        len(strict_test) > 0 and train_end < strict_test["event_time"].min()
    )
    train_sample_ok = len(train) >= MIN_TRAIN_ROWS and int(train["click"].sum()) >= MIN_TRAIN_CLICKS
    test_sample_ok = (
        len(strict_test) >= MIN_TEST_ROWS
        and int(strict_test["click"].sum()) >= MIN_TEST_CLICKS
    )

    gates = {
        # Validate source records so consolidation cannot hide malformed input rows.
        "null_train_time": int(train_raw["event_time"].isna().sum()),
        "null_test_time": int(test_raw["event_time"].isna().sum()),
        "bad_train_click": bad_clicks(train_raw),
        "bad_test_click": bad_clicks(test_raw),
        "invalid_train_payprice": invalid_costs(train_raw),
        "invalid_test_payprice": invalid_costs(test_raw),
        "advertiser_mismatch_train": advertiser_mismatches(train_raw),
        "advertiser_mismatch_test": advertiser_mismatches(test_raw),
        "null_train_bidid": train_dedup["null_bidids"],
        "null_test_bidid": test_dedup["null_bidids"],
        "hard_train_bidid_identity_conflicts": train_dedup["hard_identity_conflicts"],
        "hard_test_bidid_identity_conflicts": test_dedup["hard_identity_conflicts"],
        "strict_cross_split_bidid_overlap": int(strict_cross_bid_overlap),
    }
    structural_ok = all(value == 0 for value in gates.values())
    eligible = bool(structural_ok and chronology_ok and train_sample_ok and test_sample_ok)
    failures = [name for name, value in gates.items() if value != 0]
    if not chronology_ok:
        failures.append("strict_chronology")
    if not train_sample_ok:
        failures.append("train_sample_threshold")
    if not test_sample_ok:
        failures.append("strict_test_sample_threshold")

    row = {
        "advertiser": advertiser,
        "source_train_rows": len(train_source),
        "source_train_clicks": int(pd.to_numeric(train_source["click"], errors="coerce").sum()),
        "clean_train_rows": len(train),
        "train_duplicate_bidid_groups": train_dedup["duplicate_bidid_groups"],
        "train_duplicate_rows_removed": train_dedup["duplicate_bidid_excess_rows"],
        "train_duplicate_bidids_with_disagreement": train_dedup[
            "duplicate_bidids_with_any_disagreement"
        ],
        "train_click_disagreements_resolved": train_dedup["click_disagreements"],
        "train_payprice_disagreements_resolved": train_dedup["payprice_disagreements"],
        "train_timestamp_disagreements_resolved": train_dedup["timestamp_disagreements"],
        "train_clicks": int(train["click"].sum()),
        "source_test_rows": len(test_source),
        "source_test_clicks": int(pd.to_numeric(test_source["click"], errors="coerce").sum()),
        "deduplicated_test_rows": len(test),
        "test_duplicate_bidid_groups": test_dedup["duplicate_bidid_groups"],
        "test_duplicate_rows_removed": test_dedup["duplicate_bidid_excess_rows"],
        "test_duplicate_bidids_with_disagreement": test_dedup[
            "duplicate_bidids_with_any_disagreement"
        ],
        "test_click_disagreements_resolved": test_dedup["click_disagreements"],
        "test_payprice_disagreements_resolved": test_dedup["payprice_disagreements"],
        "test_timestamp_disagreements_resolved": test_dedup["timestamp_disagreements"],
        "strict_test_rows": len(strict_test),
        "strict_test_clicks": int(strict_test["click"].sum()),
        "train_start": train["event_time"].min(),
        "train_end": train_end,
        "strict_test_start": strict_test["event_time"].min() if len(strict_test) else pd.NaT,
        "strict_test_end": strict_test["event_time"].max() if len(strict_test) else pd.NaT,
        "overlap_rows_removed": overlap_rows_removed,
        "strict_cutoff": strict_cutoff,
        "train_sha256": sha256_file(train_path),
        "test_sha256": sha256_file(test_path),
        **gates,
        "chronology_ok": chronology_ok,
        "train_sample_ok": train_sample_ok,
        "strict_test_sample_ok": test_sample_ok,
        "eligible": eligible,
        "reason": "ok" if eligible else ", ".join(failures),
    }
    train_profile = train_profile.assign(advertiser=advertiser, split="train")
    test_profile = test_profile.assign(advertiser=advertiser, split="test")
    duplicate_profile = pd.concat([train_profile, test_profile], ignore_index=True)
    return row, train, strict_test, duplicate_profile


audits, RAW, duplicate_profiles = [], {}, []
for advertiser, folder in CAMPAIGN_DIRS.items():
    row, train, strict_test, duplicate_profile = campaign_audit(advertiser, folder)
    audits.append(row)
    if train is not None:
        RAW[advertiser] = (train, strict_test)
    if len(duplicate_profile):
        duplicate_profiles.append(duplicate_profile)

audit_df = pd.DataFrame(audits).sort_values("advertiser").reset_index(drop=True)
display(audit_df)

# Hashes are deliberately displayed in full rather than stored only in memory.
hash_columns = ["advertiser", "train_sha256", "test_sha256"]
display(audit_df[[column for column in hash_columns if column in audit_df]])

if duplicate_profiles:
    duplicate_resolution_df = pd.concat(duplicate_profiles, ignore_index=True)
    duplicate_resolution_df = duplicate_resolution_df[
        [
            "advertiser", "split", "field", "duplicate_bidids_with_disagreement",
            "resolution", "hard_failure",
        ]
    ].sort_values(["advertiser", "split", "field"]).reset_index(drop=True)
else:
    duplicate_resolution_df = pd.DataFrame(
        columns=[
            "advertiser", "split", "field", "duplicate_bidids_with_disagreement",
            "resolution", "hard_failure",
        ]
    )
display(duplicate_resolution_df)

ELIGIBLE_ADVERTISERS = audit_df.loc[audit_df["eligible"], "advertiser"].astype(int).tolist()
if len(ELIGIBLE_ADVERTISERS) != 2:
    raise AssertionError(
        "Both declared advertisers must pass eligibility after auction consolidation.\n"
        + audit_df[["advertiser", "eligible", "reason"]].to_string(index=False)
    )
if not SMOKE_TEST:
    assert set(ELIGIBLE_ADVERTISERS) == set(TARGET_ADVERTISERS)
print(f"Eligible advertisers: {ELIGIBLE_ADVERTISERS}")


## 4. Chronological development splits

The outer split preserves the original fit/calibration/validation roles. The tuning routine then
takes a recent, capped window from the fit portion and creates another chronological
tuning-train/tuning-evaluation boundary. No holdout rows participate in model or calibrator
selection.


In [ ]:
def timestamp_group_endpoints(sorted_frame):
    """Return row boundaries that never split observations sharing one timestamp."""
    times = sorted_frame["event_time"]
    if times.isna().any():
        raise ValueError("Chronological splitting requires non-null event_time values.")
    if not times.is_monotonic_increasing:
        raise ValueError("timestamp_group_endpoints expects event_time-sorted data.")
    return np.flatnonzero(times.ne(times.shift(-1)).to_numpy()) + 1


def nearest_timestamp_boundary(endpoints, target_index, lower_exclusive=0, upper_exclusive=None):
    """Choose the valid timestamp-group boundary nearest a requested row index."""
    endpoints = np.asarray(endpoints, dtype=int)
    if upper_exclusive is None:
        upper_exclusive = int(endpoints.max())
    candidates = endpoints[
        (endpoints > int(lower_exclusive)) & (endpoints < int(upper_exclusive))
    ]
    if not len(candidates):
        raise ValueError(
            "No valid timestamp boundary exists in the requested interval; "
            "inspect timestamp parsing and unique-time counts."
        )
    return int(candidates[np.argmin(np.abs(candidates - int(target_index)))])


def chronological_split(frame, fit_frac=FIT_FRAC, cal_frac=CAL_FRAC):
    data = frame.sort_values("event_time", kind="mergesort").reset_index(drop=True)
    n_rows = len(data)
    endpoints = timestamp_group_endpoints(data)
    if n_rows < 3 or len(endpoints) < 3:
        raise ValueError("Need at least three distinct timestamps for development splits.")

    # Leave at least two timestamp groups after fit and one after calibration.
    fit_candidates = endpoints[:-2]
    fit_end = int(fit_candidates[np.argmin(np.abs(fit_candidates - int(n_rows * fit_frac)))])
    cal_end = nearest_timestamp_boundary(
        endpoints,
        int(n_rows * (fit_frac + cal_frac)),
        lower_exclusive=fit_end,
        upper_exclusive=n_rows,
    )
    fit = data.iloc[:fit_end].copy()
    calibration = data.iloc[fit_end:cal_end].copy()
    validation = data.iloc[cal_end:].copy()
    assert fit.event_time.max() < calibration.event_time.min()
    assert calibration.event_time.max() < validation.event_time.min()
    return fit, calibration, validation


def chronological_tuning_window(fit_frame, max_rows=TUNING_WINDOW_MAX_ROWS, eval_frac=TUNING_EVAL_FRAC):
    """Use a recent fit-only window and split at the nearest complete timestamp group."""
    data = fit_frame.sort_values("event_time", kind="mergesort").reset_index(drop=True)
    if len(data) > max_rows:
        # Include the whole timestamp group at the cap edge instead of truncating it.
        cutoff_time = data.iloc[-max_rows]["event_time"]
        data = data.loc[data["event_time"] >= cutoff_time].reset_index(drop=True)
    endpoints = timestamp_group_endpoints(data)
    if len(endpoints) < 2:
        raise ValueError(
            "The tuning window contains fewer than two distinct timestamps; "
            "verify millisecond timestamp parsing or increase TUNING_WINDOW_MAX_ROWS."
        )
    target = int(round(len(data) * (1 - eval_frac)))
    boundary = nearest_timestamp_boundary(
        endpoints, target, lower_exclusive=0, upper_exclusive=len(data)
    )
    tune_train = data.iloc[:boundary].copy()
    tune_eval = data.iloc[boundary:].copy()
    assert tune_train.event_time.max() < tune_eval.event_time.min()
    return tune_train, tune_eval


split_summary = []
for advertiser in ELIGIBLE_ADVERTISERS:
    fit, calibration, validation = chronological_split(RAW[advertiser][0])
    tune_train, tune_eval = chronological_tuning_window(fit)
    strict_test = RAW[advertiser][1]
    assert validation.event_time.max() < strict_test.event_time.min()
    split_summary.append(
        {
            "advertiser": advertiser,
            "fit_rows": len(fit), "fit_clicks": int(fit.click.sum()),
            "tuning_train_rows": len(tune_train),
            "tuning_eval_rows": len(tune_eval),
            "tuning_train_end": tune_train.event_time.max(),
            "tuning_eval_start": tune_eval.event_time.min(),
            "tuning_window_unique_timestamps": int(
                pd.concat([tune_train.event_time, tune_eval.event_time]).nunique()
            ),
            "calibration_rows": len(calibration),
            "calibration_clicks": int(calibration.click.sum()),
            "validation_rows": len(validation),
            "validation_clicks": int(validation.click.sum()),
            "strict_test_rows": len(strict_test),
            "strict_test_clicks": int(strict_test.click.sum()),
        }
    )
display(pd.DataFrame(split_summary))


## Stage hand-off

Only eligible, consolidated, chronologically valid data is persisted. Notebook 02 reads
these files instead of re-running or silently changing the audit.


In [ ]:
PREPARED_DIR = ARTIFACT_ROOT / "01_prepared"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

# Parquet keeps timestamps and types stable while compressing the multi-million-row logs.
try:
    import pyarrow  # noqa: F401
except ImportError as exc:
    raise ImportError("Notebook 01 requires pyarrow for the stage hand-off files.") from exc

for advertiser in ELIGIBLE_ADVERTISERS:
    train, strict_test = RAW[advertiser]
    train.sort_values("event_time", kind="mergesort").to_parquet(
        PREPARED_DIR / f"advertiser_{advertiser}_train.parquet",
        index=False,
        compression="zstd",
    )
    strict_test.sort_values("event_time", kind="mergesort").to_parquet(
        PREPARED_DIR / f"advertiser_{advertiser}_strict_test.parquet",
        index=False,
        compression="zstd",
    )

audit_df.to_csv(PREPARED_DIR / "data_audit.csv", index=False)
duplicate_resolution_df.to_csv(PREPARED_DIR / "duplicate_resolution.csv", index=False)
pd.DataFrame(split_summary).to_csv(PREPARED_DIR / "split_summary.csv", index=False)

data_manifest = {
    "schema_version": NOTEBOOK_SCHEMA_VERSION,
    "data_root": str(DATA_ROOT),
    "eligible_advertisers": [int(x) for x in ELIGIBLE_ADVERTISERS],
    "strict_temporal_holdout": STRICT_TEMPORAL_HOLDOUT,
    "timestamp_format": "yyyyMMddHHmmssSSS",
    "train_sha256": {
        str(row.advertiser): row.train_sha256 for row in audit_df.itertuples()
    },
    "test_sha256": {
        str(row.advertiser): row.test_sha256 for row in audit_df.itertuples()
    },
}
(PREPARED_DIR / "data_manifest.json").write_text(
    json.dumps(data_manifest, indent=2), encoding="utf-8"
)
print(f"Prepared artifacts written to {PREPARED_DIR}")
